In [3]:
# ORM_Exam.ipynb

## Jupyter Lab에서 Django shell을 실행하기 위한 설정 / in ipynb
import os
import django
# 환경변수로 config/settings.py의 위치를 설정
os.environ['DJANGO_SETTINGS_MODULE'] = 'config.settings'
os.environ['DJANGO_ALLOW_ASYNC_UNSAFE'] = 'true'

django.setup()

In [4]:
# 조회 테스트
from polls.models import Question, Choice

q = Question.objects.all()
q

<QuerySet [<Question: 1. 좋아하는 색이 무엇입니까?>]>


### 조회
- ModelClass.objects -> Model Manager를 반환.
- ModelManager : SQL 작업을 할 수 있는 메소드를 제공해주는 객체

### 조회 메소드
- `all()` : 전체 조회
- `filter()`, `exclude()` : 조건으로 조회(where 절)
- `get()` : 조회 결과가 하나인 조건으로 조회(PK로 조회)

### 조회 결과
- `QuerySet` 객체 : 조회 결과가 여러 개일때 QuerySet에 모아서 반환.
    - 조회결과를 바탕으로 추가 DB 작업을 진행할 수 있다.
    - 개별 데이터는 Model 객체에 담아서 반환.
- Model 객체 : 조회결과가 하나 (`get()`) 일 때


In [5]:
from polls.models import Question, Choice

In [6]:
model_manager = Question.objects
type(model_manager)

django.db.models.manager.Manager

In [7]:
result = model_manager.all()
print('조회한 데이터 개수 : ', len(result))
print('all()로 실행된 SQL문을 조회')
print(result.query)

조회한 데이터 개수 :  1
all()로 실행된 SQL문을 조회
SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question"


In [8]:
print(type(result))

<class 'django.db.models.query.QuerySet'>


In [9]:
# QuerySet -> Iterable
for q in result:
    # print(type(q))
    print(q, q.pub_date)

1. 좋아하는 색이 무엇입니까? 2025-07-07 05:30:42.105229+00:00


In [12]:
# QuerySet -> Subscriptable
q = result[0]
print(q.id, q.pk)
print(q.question_text, q.pub_date)
print(type(q.pk), type(q.question_text), type(q.pub_date))

1 1
좋아하는 색이 무엇입니까? 2025-07-07 05:30:42.105229+00:00
<class 'int'> <class 'str'> <class 'datetime.datetime'>


In [13]:
# QuerySet - 첫 번쨰, 마지막 index 값 조회
q_s = result.first()
q_e = result.last()
q_s.pk, q_e.pk

(1, 1)

In [14]:
# 음수 indexing은 지원하지 않는다.

In [15]:
# QuerySet을 이용해서 정렬 (order by) - QS.orderby('기준 Field명')
# result.order_by('questiont_text')
result.order_by('-question_text')

<QuerySet [<Question: 1. 좋아하는 색이 무엇입니까?>]>

In [21]:
# Choice에 모든 데이터를 조회 -> choice_text 기준으로 정렬
results = Choice.objects.all().order_by('choice_text', '-votes')
for c in results:
    print(c.pk, c.choice_text, c.votes)

1 빨강 10
2 파랑 5


In [22]:
print(results.query)

SELECT "polls_choice"."id", "polls_choice"."choice_text", "polls_choice"."votes", "polls_choice"."question_id" FROM "polls_choice" ORDER BY "polls_choice"."choice_text" ASC, "polls_choice"."votes" DESC


### Where 절
- filter()
    - 조회 조건이 True인 행들을 조회 -> QuerySet을 반환
- exclude()
    - 조회 조건이 False인 행들을 조회 -> QuerySet을 반환
- get()
    - 조회 조건이 True인 행이 1개 일때 조회 -> Model에 결과를 담아서 반환.
    - 조회 결과가 2행 이상이거나 없을 경우 Exception 발생 
- **조회조건**
    - `Field이름__비교연산자 = 비교할 값`

In [24]:
# PK 조회 -> 결과 1행 | 0행
result = Question.objects.get(pk=1)     # where id=1. 동등 비교 : field명 = 값
result = Question.objects.filter(pk=1)  # QuerySet에 담아줘
print(type(result))
print(result)

<class 'django.db.models.query.QuerySet'>
<QuerySet [<Question: 1. 좋아하는 색이 무엇입니까?>]>


In [26]:
# 비교 연산
result = Question.objects.filter(pk__lt=5)
print(result)

<QuerySet [<Question: 1. 좋아하는 색이 무엇입니까?>]>


In [33]:
# 문자열 부분일치 - like (xxx 를 포함, xxx로 시작, xxx로 끝)
result = Question.objects.filter(question_text__contains='색깔')
# question_text like '%색깔%'
result = Question.objects.filter(question_text__endswith='무엇입니까?')
# question_text like '%무엇입니까?'
# result = Question.objects.filter(question_text__startswith='좋아하는?')
# question_text like '좋아하는%'

print(result.query)
print(result)

SELECT "polls_question"."id", "polls_question"."question_text", "polls_question"."pub_date" FROM "polls_question" WHERE "polls_question"."question_text" LIKE %무엇입니까? ESCAPE '\'
<QuerySet [<Question: 1. 좋아하는 색이 무엇입니까?>]>


In [34]:
# in 연산
result = Choice.objects.filter(pk__in=[1, 2, 3])
print(result)

<QuerySet [<Choice: 1. 빨강>, <Choice: 2. 파랑>]>
